# Из тренажёра — в настоящий Spark
Тренажёр Stepik позволяет изучать выбранные операции без установки большого стека.
Теперь проверим знакомое выражение в настоящем PySpark. Этот notebook можно выполнить
после базового блока DataFrame API. Здесь не нужны Kafka, MinIO и Delta. Используйте учебный Jupyter с PySpark 3.5.3
либо отдельную среду с Python 3.11, PySpark 3.5.3 и Java 17.
![Выражение и выполнение](../assets/lazy-column.png)


In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[2]").appName("first-real-spark").getOrCreate()
df = spark.createDataFrame([(1,100),(2,None),(3,250)], "id long, amount long")
result = df.withColumn("band", F.when(F.col("amount").isNull(), "unknown")
                       .when(F.col("amount") >= 200, "large").otherwise("regular"))
result.orderBy("id").show()


Ожидаемые пары id/band: 1/regular, 2/unknown, 3/large.
Сортировка здесь задана явно. Без orderBy не следует рассчитывать на порядок строк.


In [ ]:
assert [(r.id,r.band) for r in result.orderBy("id").collect()] == [(1,"regular"),(2,"unknown"),(3,"large")]
result.printSchema()
result.explain(mode="formatted")


Объясни: что вернул when? Что запустило вычисление? Где задан порядок результата?
Не нужно запоминать все узлы плана. Найди чтение исходных строк и вычисление нового столбца.
Авторский `spark_api.py` помогает отрабатывать поддерживаемое подмножество выражений.
Планы, физический кэш и работу распределённого движка изучаем в настоящем Spark.


## Необязательно после урока 8.5: пометка и заполнение кэша

Выполняйте ячейки по очереди. Сначала только пометим таблицу — без action.
Это продолжение знакомого упражнения с `is_cached`, без новых сервисов.

![Пометка, вычисление и освобождение кэша](../assets/cache.png)


In [ ]:
cached = (
    spark.range(0, 10_000, numPartitions=2)
         .where(F.col("id") % 2 == 0)
         .cache()
)
print("Пометка до action:", cached.is_cached)  # True


Теперь вызовем action. В таблице 5 000 чётных чисел.
Первый `count()` вычисляет весь этот результат; второй может использовать сохранённые блоки.
В плане найдите `InMemoryTableScan`. Наличие узла описывает способ чтения, а заполнение
кэша можно наблюдать во вкладке Storage в Spark UI до следующей ячейки освобождения.
Ссылка на Spark UI зависит от вашей среды.


In [ ]:
print("Первый count:", cached.count())
print("Повторный count:", cached.count())
cached.explain(mode="formatted")


Оба ответа должны быть 5 000. Равенство ответов проверяет результат вычисления,
но не доказывает ускорение. На таком маленьком примере не сравниваем быстродействие.

Почему флаг был True ещё до первого count? Почему show(5) не гарантирует
заполнения всего кэша? Закончив наблюдение, освободите сохранённые блоки:


In [ ]:
cached.unpersist(blocking=True)
print("Пометка после unpersist:", cached.is_cached)  # False


`blocking=True` здесь позволяет дождаться удаления блоков перед продолжением
эксперимента. Обычный `unpersist()` по умолчанию не ждёт завершения удаления.

Справка: [cache](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.cache.html),
[unpersist](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.unpersist.html).
